In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: функція Qiskit від ColibriTD
*Дивись [довідник API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Функції Qiskit — це експериментальна можливість, доступна користувачам IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API) Plan. Вони перебувають у статусі попереднього релізу та можуть змінюватись.
## Огляд
Розв'язувач диференціальних рівнянь у частинних похідних (PDE), представлений тут, є частиною нашої платформи Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) і запакований як функція Qiskit. За допомогою функції QUICK-PDE ти можеш розв'язувати предметно-орієнтовані диференціальні рівняння в частинних похідних на QPU IBM Quantum. Ця функція базується на алгоритмі, описаному в [статті ColibriTD про H-DES](https://arxiv.org/abs/2410.01130). Цей алгоритм може розв'язувати складні мультифізичні задачі, починаючи з обчислювальної гідродинаміки (CFD) та деформації матеріалів (MD), і незабаром з'являться нові варіанти використання.

Щоб впоратися з диференціальними рівняннями, пробні розв'язки кодуються як лінійні комбінації ортогональних функцій (зазвичай поліноми Чебишева, і конкретніше $2^n$ таких, де $n$ — кількість кубітів, що кодують твою функцію), параметризовані кутами Змінного Квантового Ланцюга (VQC). Анзац генерує стан, що кодує функцію, яку обчислюють спостережувані величини, комбінації яких дозволяють обчислити функцію у всіх точках. Потім ти можеш обчислити функцію втрат, у якій закодовано диференціальні рівняння, та налаштувати кути в гібридному циклі, як показано нижче. Пробні розв'язки поступово наближаються до фактичних розв'язків, доки не буде досягнуто задовільного результату.

![Робочий процес функції QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

На додаток до цього гібридного циклу, ти також можеш поєднувати різні оптимізатори в ланцюжок. Це корисно, коли потрібно, щоб глобальний оптимізатор знайшов хороший набір кутів, а потім точніший оптимізатор пішов за градієнтом до найкращого набору сусідніх кутів. У випадку обчислювальної гідродинаміки (CFD) стандартна послідовність оптимізації дає найкращі результати — але у випадку деформації матеріалів (MD), хоча стандартні налаштування і дають хороші результати, ти можеш налаштувати її додатково для переваг, специфічних для конкретної задачі.

Зауваж, що для кожної змінної функції ми вказуємо кількість кубітів (з якою ти можеш експериментувати). Стекуючи 10 ідентичних схем та обчислюючи 10 ідентичних спостережуваних на різних кубітах в одній великій схемі, ти можеш знижувати шуми в процесі CMA-оптимізації, спираючись на метод навчання шумів, та значно зменшити потрібну кількість знімків (shots).

### Обчислювальна гідродинаміка
Невʼязке рівняння Бюргерса моделює потік невʼязкої рідини таким чином:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ представляє поле швидкості рідини. Цей варіант використання має часову граничну умову: ти можеш задати початкову умову, а потім дозволити системі розслабитись. Наразі єдиними прийнятними початковими умовами є лінійні функції: $ax + b$.

Аргументи диференціальних рівнянь CFD знаходяться на фіксованій сітці, таким чином:

- $t$ знаходиться між 0 і 0.95 з 30 точками вибірки. $x$ знаходиться між 0 і 0.95 з кроком 0.2375.

### Деформація матеріалів
Цей варіант використання зосереджений на гіпопружній деформації з одновимірним випробуванням на розтяг, у якому стержень, закріплений в просторі, тягнуть за інший кінець. Ми описуємо задачу таким чином:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ представляє об'ємний модуль матеріалу, що розтягується, $n$ — показник степеневого закону, $b$ — силу на одиницю маси, $\epsilon_0$ — межу пропорційного напруження, $\sigma_0$ — межу пропорційної деформації, $u$ — функцію напруження, а $\sigma$ — функцію деформації.

Розглянутий стержень має одиничну довжину. Цей варіант використання має граничну умову для поверхневого напруження $t$, тобто кількість роботи, необхідної для розтягування стержня.

Аргументи диференціальних рівнянь MD знаходяться на фіксованій сітці, таким чином:

- $x$ знаходиться між 0 і 1 з кроком 0.04.

## Бенчмарки
У наступній таблиці представлено статистику різних запусків нашої функції.

| Приклад                            | Кількість кубітів | Ініціалізація         | Похибка   | Загальний час (хв) | Використання середовища виконання (хв) |
| ---------------------------------- | ----------------- | --------------------- | --------- | ------------------ | --------------------------------------- |
| Невʼязке рівняння Бюргерса         | 50                | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                 | 25                                      |
| Гіпопружне одновимірне розтягнення | 18                | `RANDOM`              | $10^{-2}$ | 123                | 100                                     |

## Початок роботи
Заповни [форму для запиту доступу до функції QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Потім, якщо ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі, вибери функцію таким чином:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_cloud / ibm_quantum_platform",
    instance="USER_CRN / HGP",
    token="USER_API_KEY / IQP_API_TOKEN",
)

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

In [ ]:
quick = catalog.load("colibritd/quick-pde")

Перевір [статус](/guides/functions#check-job-status) або отримай [результати](/guides/functions#retrieve-results) навантаження своєї функції Qiskit таким чином:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(
    use_case="CFD_BURGER", physical_parameters={"a": 1.0, "b": 0.0}
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [ ]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())
solution = job.result()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_result_3d(result):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    t, x = np.meshgrid(result["samples"]["t"], result["samples"]["x"])

    ax.plot_surface(
        t,
        x,
        result["functions"]["u"],
        edgecolor="royalblue",
        lw=0.25,
        rstride=26,
        cstride=26,
        alpha=0.3,
    )
    ax.scatter(t, x, result["functions"]["u"], marker=".")
    ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

    plt.show()


# Call
plot_result_3d(solution)

![Вихідні дані попередньої комірки коду](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Деформація матеріалів
Варіант використання деформації матеріалів вимагає фізичних параметрів твого матеріалу та прикладеної сили, таким чином:

In [ ]:
# Launches the solving for an arbitrary mu
job = quick.run(use_case="CFD_EULER", physical_parameters={"mu": 0.1})

solution = job.result()


# Colorplot function
def plot_result_2d(result):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    configs = {
        "g": {"cmap": "viridis", "title": "g(t, x)"},
        "u": {"cmap": "plasma", "title": "u(t, x)"},
    }

    t = result["samples"]["t"]
    x = result["samples"]["x"]

    for ax, (field, cfg) in zip(axes, configs.items()):
        v = result["functions"][field]

        im = ax.contourf(t, x, v, levels=50, cmap=cfg["cmap"])
        fig.colorbar(im, ax=ax, label=cfg["title"])

        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title(cfg["title"], fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_result_2d(solution)

![Вихідні дані попередньої комірки коду](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Нижче наведено приклад того, як отримати значення функції для конкретного набору координат:

In [ ]:
# Select the properties of your material
job = quick.run(
    use_case="MD",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# Plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

## Отримання повідомлень про помилки
Якщо статус твого навантаження — `ERROR`, використай `job.error_message()`, щоб отримати повідомлення про помилку для налагодження, таким чином:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Fetch error messages

If your workload status is `ERROR`, use `job.error_message()` to fetch the error message to help debug, as follows:

In [ ]:
job = quick.run(use_case="MD", physical_params={})

print(job.error_message())


# A wrapper can also be used for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

## Отримати підтримку

Для підтримки звертайся на qiskit-function-support@colibritd.com.

## Наступні кроки

> **Tip:** - Заповни форму для [запиту доступу до функції QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK).
> - Відвідай [довідник API](https://docs.quantum.ibm.com/api/functions/colibritd-pde) для цієї функції Qiskit.
> - Спробуй змоделювати потік невʼязкої рідини за допомогою QUICK-PDE у [підручнику](/tutorials/colibritd-pde).
> - Ознайомся з [Jaffali, H., та ін. (2025). H-DES: a Quantum-Classical Hybrid Differential Equation Solver. arXiv preprint arXiv:2410.01130](https://arxiv.org/abs/2410.01130).